In [1]:
import pandas as pd
import seaborn as sns
import numpy as np

In [3]:
rent_data = pd.read_csv("data/raw/house_rent_dataset.csv")

In [4]:
print(rent_data.head())

    Posted On  BHK   Rent  Size            Floor    Area Type  \
0  2022-05-18    2  10000  1100  Ground out of 2   Super Area   
1  2022-05-13    2  20000   800       1 out of 3   Super Area   
2  2022-05-16    2  17000  1000       1 out of 3   Super Area   
3  2022-07-04    2  10000   800       1 out of 2   Super Area   
4  2022-05-09    2   7500   850       1 out of 2  Carpet Area   

              Area Locality     City Furnishing Status  Tenant Preferred  \
0                    Bandel  Kolkata       Unfurnished  Bachelors/Family   
1  Phool Bagan, Kankurgachi  Kolkata    Semi-Furnished  Bachelors/Family   
2   Salt Lake City Sector 2  Kolkata    Semi-Furnished  Bachelors/Family   
3               Dumdum Park  Kolkata       Unfurnished  Bachelors/Family   
4             South Dum Dum  Kolkata       Unfurnished         Bachelors   

   Bathroom Point of Contact  
0         2    Contact Owner  
1         1    Contact Owner  
2         1    Contact Owner  
3         1    Contact Owner

In [99]:
rent_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4746 entries, 0 to 4745
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Posted On          4746 non-null   object
 1   BHK                4746 non-null   int64 
 2   Rent               4746 non-null   int64 
 3   Size               4746 non-null   int64 
 4   Floor              4746 non-null   object
 5   Area Type          4746 non-null   object
 6   Area Locality      4746 non-null   object
 7   City               4746 non-null   object
 8   Furnishing Status  4746 non-null   object
 9   Tenant Preferred   4746 non-null   object
 10  Bathroom           4746 non-null   int64 
 11  Point of Contact   4746 non-null   object
dtypes: int64(4), object(8)
memory usage: 445.1+ KB


In [109]:
# cleanup
rent_data = rent_data.drop(columns=["Posted On", "Point of Contact", "Area Locality"])

In [110]:
rent_data

,BHK,Rent,Size,Floor,Area Type,City,Furnishing Status,Tenant Preferred,Bathroom
0,2,10000,1100,Ground out of 2,Super Area,Kolkata,Unfurnished,Bachelors/Family,2
1,2,20000,800,1 out of 3,Super Area,Kolkata,Semi-Furnished,Bachelors/Family,1
2,2,17000,1000,1 out of 3,Super Area,Kolkata,Semi-Furnished,Bachelors/Family,1
3,2,10000,800,1 out of 2,Super Area,Kolkata,Unfurnished,Bachelors/Family,1
4,2,7500,850,1 out of 2,Carpet Area,Kolkata,Unfurnished,Bachelors,1
...,...,...,...,...,...,...,...,...,...
4741,2,15000,1000,3 out of 5,Carpet Area,Hyderabad,Semi-Furnished,Bachelors/Family,2
4742,3,29000,2000,1 out of 4,Super Area,Hyderabad,Semi-Furnished,Bachelors/Family,3
4743,3,35000,1750,3 out of 5,Carpet Area,Hyderabad,Semi-Furnished,Bachelors/Family,3
4744,3,45000,1500,23 out of 34,Carpet Area,Hyderabad,Semi-Furnished,Family,2


In [111]:
from sklearn.preprocessing import OneHotEncoder

cols = ["Furnishing Status", "Tenant Preferred", "Area Type", "City"]

ohe = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")

encoded = ohe.fit_transform(rent_data[cols])
encoded_data = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(cols), index=rent_data.index)

rent_data = pd.concat([rent_data.drop(columns=cols), encoded_data], axis=1)
rent_data

,BHK,Rent,Size,Floor,Bathroom,Furnishing Status_Semi-Furnished,Furnishing Status_Unfurnished,Tenant Preferred_Bachelors/Family,Tenant Preferred_Family,Area Type_Carpet Area,Area Type_Super Area,City_Chennai,City_Delhi,City_Hyderabad,City_Kolkata,City_Mumbai
0,2,10000,1100,Ground out of 2,2,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
1,2,20000,800,1 out of 3,1,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2,2,17000,1000,1 out of 3,1,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
3,2,10000,800,1 out of 2,1,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
4,2,7500,850,1 out of 2,1,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4741,2,15000,1000,3 out of 5,2,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
4742,3,29000,2000,1 out of 4,3,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
4743,3,35000,1750,3 out of 5,3,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
4744,3,45000,1500,23 out of 34,2,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0


In [113]:
def extract_floor_info(floor_str):
    floor_str = str(floor_str).strip()

    if "out of" in floor_str.lower():

        current, total = floor_str.split(" out of ")

        current = current.strip().lower()

        if current == "ground":
            current = 0
        elif current == "upper basement":
            current = 0
        elif current == "lower basement":
            current = -1
        else:
            current = int(current)

        total = int(total)

        return pd.Series([current, total])

    elif floor_str.lower() == "ground":
        return pd.Series([0, 0])

    elif floor_str.isdigit():
        floor_num = int(floor_str)
        return pd.Series([floor_num, floor_num])

    else:
        print("Unhandled:", floor_str)
        return pd.Series([np.nan, np.nan])

rent_data[["Current Floor", "Total Floors"]] = (
    rent_data["Floor"]
    .apply(extract_floor_info)
)

rent_data.drop("Floor", axis=1, inplace=True)

In [117]:
X = rent_data.drop(columns=["Rent"])
y = rent_data["Rent"]

In [118]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [125]:
X_train

,BHK,Size,Bathroom,Furnishing Status_Semi-Furnished,Furnishing Status_Unfurnished,Tenant Preferred_Bachelors/Family,Tenant Preferred_Family,Area Type_Carpet Area,Area Type_Super Area,City_Chennai,City_Delhi,City_Hyderabad,City_Kolkata,City_Mumbai,Current Floor,Total Floors
1995,2,950,2,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,6,8
1497,2,800,2,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,3,4
2763,1,545,1,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,2,3
1351,1,465,1,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,3,4
1862,2,900,2,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,4,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4426,3,1500,3,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1,2
466,3,1200,2,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,3,3
3092,2,800,2,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,13,17
3772,3,3500,3,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0,1


In [120]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

In [121]:
y_pred = model.predict(X_test)

In [122]:
from sklearn.metrics import r2_score

r2 = r2_score(y_test, y_pred)
r2

0.5201595940170398

In [123]:
print("Train R²:", model.score(X_train, y_train))
print("Test R²:", model.score(X_test, y_test))

Train R²: 0.2911597471227324
Test R²: 0.5201595940170398


#Underfitting case

In [128]:
print(rent_data.corr(numeric_only=True)["Rent"].sort_values(ascending=False))

Rent                                 1.000000
Bathroom                             0.441215
Size                                 0.413551
BHK                                  0.369718
Total Floors                         0.352268
City_Mumbai                          0.327038
Current Floor                        0.326898
Area Type_Carpet Area                0.215769
Tenant Preferred_Family              0.063941
Furnishing Status_Semi-Furnished     0.045309
City_Delhi                          -0.027072
Tenant Preferred_Bachelors/Family   -0.078774
City_Chennai                        -0.082361
City_Hyderabad                      -0.087465
City_Kolkata                        -0.105322
Furnishing Status_Unfurnished       -0.126271
Area Type_Super Area                -0.215499
Name: Rent, dtype: float64
